# 07 — DivKG REINFORCE Fine-Tuning

**Purpose:** Fine-tune the SRD-pretrained GRU4Rec model using **REINFORCE**
with a KL divergence penalty — Method 2 of the BDC two-stage recommender
training pipeline.

## Theory: DivKG REINFORCE

Two policies are maintained:
- **Reference policy** $\pi_{\text{ref}}$ — the SRD-pretrained model (frozen)
- **Trainable policy** $\pi_{\theta}$ — initialized from the same checkpoint

The RL objective maximizes expected reward while penalizing divergence from the reference:

$$\mathcal{J}(\theta) = \mathbb{E}_{a \sim \pi_\theta}[R(a)] - \beta \cdot D_{KL}(\pi_\theta \| \pi_{\text{ref}})$$

REINFORCE gradient estimate:
$$\nabla_\theta \mathcal{J} = \mathbb{E}[(R - b) \nabla_\theta \log \pi_\theta(a | s)]$$

where $b$ is a baseline (mean reward) for variance reduction.

## Reward Function

$$R = \text{nDCG@5} + 0.2 \times \text{novelty\_score}$$

The novelty score rewards recommending long-tail items that the user hasn't
frequently seen — encouraging knowledge exploration.

In [ ]:
import os
import sys
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.abspath('..'))

OUTPUT_DIR = '../output/slates'
SRD_CHECKPOINT = os.path.join(OUTPUT_DIR, 'srd_model.pt')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. Reward Function

$$R = \text{nDCG@5} + 0.2 \times \text{novelty\_score}$$

In [ ]:
def compute_ndcg_at_k(recommended: list, actual: list, k: int = 5) -> float:
    """Compute nDCG@K for a single user."""
    act_set = set(actual)
    if not act_set:
        return 0.0
    idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(act_set), k)))
    if idcg == 0:
        return 0.0
    dcg = sum(1.0 / np.log2(idx + 2)
              for idx, item in enumerate(recommended[:k])
              if item in act_set)
    return dcg / idcg


def compute_novelty_score(recommended: list, item_popularity: dict) -> float:
    """Novelty = mean(-log2(p(item))) — higher is more novel (long-tail)."""
    if not recommended:
        return 0.0
    return np.mean([-np.log2(item_popularity.get(item, 1e-9)) for item in recommended])


def compute_rewards(
    recommended_slates: list,
    ground_truth_list: list,
    item_popularity: dict,
    k: int = 5,
    novelty_weight: float = 0.2,
) -> torch.Tensor:
    """
    Compute composite reward for a batch of recommendation slates.

    Parameters
    ----------
    recommended_slates : list of list of int
        Batch of recommended item ID lists.
    ground_truth_list : list of list of int
        Batch of ground truth item ID lists.
    item_popularity : dict mapping item_id -> float
        Item popularity distribution.
    k : int
        nDCG cutoff.
    novelty_weight : float
        Weight on novelty component (default: 0.2).

    Returns
    -------
    torch.Tensor of shape (batch,) — one reward per sample
    """
    rewards = []
    for slate, actual in zip(recommended_slates, ground_truth_list):
        ndcg    = compute_ndcg_at_k(slate, actual, k)
        novelty = compute_novelty_score(slate, item_popularity)
        r = ndcg + novelty_weight * novelty
        rewards.append(r)
    return torch.tensor(rewards, dtype=torch.float32)


print('compute_rewards() defined.')
print('R = nDCG@5 + 0.2 * novelty_score')

## 2. REINFORCE Training Step with KL Divergence Penalty

In [ ]:
def train_rl_step(
    policy_model: nn.Module,
    ref_model: nn.Module,
    sequences: torch.Tensor,
    rewards: torch.Tensor,
    optimizer: torch.optim.Optimizer,
    kl_beta: float = 0.05,
) -> dict:
    """
    One REINFORCE gradient update step with KL divergence regularization.

    Parameters
    ----------
    policy_model : nn.Module
        Trainable policy (being fine-tuned).
    ref_model : nn.Module
        Reference policy (frozen SRD checkpoint).
    sequences : torch.Tensor  (batch, seq_len)
        Input interaction sequences.
    rewards : torch.Tensor  (batch,)
        Per-sample rewards from compute_rewards().
    optimizer : torch.optim.Optimizer
    kl_beta : float
        Weight of KL penalty term.

    Returns
    -------
    dict with 'total_loss', 'policy_loss', 'kl_loss', 'mean_reward'
    """
    policy_model.train()
    ref_model.eval()

    # Forward pass: trainable policy
    logits_policy = policy_model(sequences)          # (B, V)
    log_probs_policy = F.log_softmax(logits_policy, dim=-1)  # (B, V)

    # Forward pass: reference policy (no grad)
    with torch.no_grad():
        logits_ref = ref_model(sequences)
        probs_ref  = F.softmax(logits_ref, dim=-1)

    # REINFORCE: sample actions from policy
    probs_policy = torch.exp(log_probs_policy)
    actions = torch.multinomial(probs_policy.detach(), num_samples=1).squeeze(1)  # (B,)

    # Log probability of sampled actions under policy
    log_prob_actions = log_probs_policy[torch.arange(len(actions)), actions]  # (B,)

    # Variance reduction: subtract mean reward baseline
    baseline = rewards.mean()
    advantage = rewards.to(log_prob_actions.device) - baseline

    # REINFORCE loss: -E[(R - b) * log pi(a|s)]
    policy_loss = -(advantage * log_prob_actions).mean()

    # KL divergence penalty: KL(pi_theta || pi_ref)
    kl_loss = F.kl_div(
        log_probs_policy,
        probs_ref,
        reduction='batchmean',
        log_target=False,
    )

    total_loss = policy_loss + kl_beta * kl_loss

    optimizer.zero_grad()
    total_loss.backward()
    torch.nn.utils.clip_grad_norm_(policy_model.parameters(), max_norm=1.0)
    optimizer.step()

    return {
        'total_loss':  total_loss.item(),
        'policy_loss': policy_loss.item(),
        'kl_loss':     kl_loss.item(),
        'mean_reward': rewards.mean().item(),
    }


print('train_rl_step() defined.')

## 3. Demonstration with Synthetic Data

Shows one complete RL step using small random logits tensors.

In [ ]:
from scripts.train_srd import GRU4RecBackbone
import copy

# Small synthetic setup
DEMO_VOCAB  = 50
DEMO_BATCH  = 8
DEMO_SEQ    = 10
KL_BETA     = 0.05

# Create two identical models (trainable + reference)
policy_model = GRU4RecBackbone(vocab_size=DEMO_VOCAB, embed_dim=32, hidden_dim=64).to(device)
ref_model    = copy.deepcopy(policy_model).to(device)
for p in ref_model.parameters():
    p.requires_grad_(False)  # Freeze reference

optimizer = torch.optim.Adam(policy_model.parameters(), lr=1e-4)

# Synthetic sequences and rewards
rng = np.random.default_rng(7)
sequences_np = rng.integers(0, DEMO_VOCAB, (DEMO_BATCH, DEMO_SEQ))
sequences = torch.from_numpy(sequences_np).long().to(device)

# Simulate rewards (in practice computed from nDCG + novelty)
rewards = torch.tensor(rng.uniform(0.1, 0.9, DEMO_BATCH).astype(np.float32))

print(f'Policy model parameters:    {sum(p.numel() for p in policy_model.parameters()):,}')
print(f'Reference model (frozen):   {sum(p.numel() for p in ref_model.parameters()):,}')
print(f'Batch size: {DEMO_BATCH} | Sequence length: {DEMO_SEQ} | Vocab: {DEMO_VOCAB}')
print(f'Synthetic rewards: {rewards.numpy().round(3)}')
print()

# Execute one RL step
step_result = train_rl_step(
    policy_model, ref_model, sequences, rewards, optimizer, kl_beta=KL_BETA
)

print('=== RL Step Result ===')
for k, v in step_result.items():
    print(f'  {k:<15}: {v:.6f}')

## 4. Multi-Step RL Training Loop Demo

In [ ]:
N_RL_STEPS  = 10
step_history = []

print(f'Running {N_RL_STEPS} RL steps...')
print(f'{"Step":<6} {"Total Loss":<14} {"Policy Loss":<14} {"KL Loss":<12} {"Mean Reward"}')
print('-' * 60)

for step in range(1, N_RL_STEPS + 1):
    # In production: sample real sequences from the interaction dataset
    seq_batch = torch.from_numpy(rng.integers(0, DEMO_VOCAB, (DEMO_BATCH, DEMO_SEQ)).astype(np.int64)).to(device)
    rew_batch = torch.tensor(rng.uniform(0.1, 0.9, DEMO_BATCH).astype(np.float32))

    result = train_rl_step(policy_model, ref_model, seq_batch, rew_batch, optimizer, kl_beta=KL_BETA)
    step_history.append(result)

    if step % 2 == 0 or step == 1:
        print(f"{step:<6} {result['total_loss']:<14.6f} {result['policy_loss']:<14.6f} "
              f"{result['kl_loss']:<12.6f} {result['mean_reward']:.6f}")

print()
print('RL fine-tuning demo complete.')

## 5. Loading SRD Checkpoint for Full Training

In [ ]:
print('=== How to load SRD checkpoint for full RL fine-tuning ===')
print()
print('# Load the SRD pretrained checkpoint')
print(f'ckpt = torch.load("{SRD_CHECKPOINT}")')
print()
print('# Rebuild the model with same vocab size')
print('policy_model = GRU4RecBackbone(vocab_size=ckpt["vocab_size"])')
print('policy_model.load_state_dict(ckpt["state_dict"])')
print()
print('# Create frozen reference policy from same weights')
print('ref_model = copy.deepcopy(policy_model)')
print('for p in ref_model.parameters(): p.requires_grad_(False)')
print()

# Try actual loading if checkpoint exists
if os.path.exists(SRD_CHECKPOINT):
    ckpt = torch.load(SRD_CHECKPOINT, map_location='cpu', weights_only=False)
    print(f'Checkpoint loaded from: {SRD_CHECKPOINT}')
    print(f'  vocab_size:      {ckpt["vocab_size"]}')
    print(f'  hyperparams:     {ckpt["hyperparams"]}')
    print(f'  epoch_losses:    {ckpt.get("epoch_losses", [])}')
else:
    print(f'[NOTE] Checkpoint not found at {SRD_CHECKPOINT}')
    print('  Run notebook 06_srd_loss_training.ipynb first.')